## Data Preparation

You can download the data here: https://drive.google.com/file/d/11quShgLOlDvakyMJjMNVdNKCLgXg6kqv/view?usp=drive_link

It's from CZ biohub.

In [ ]:
import anndata as ad

atlas = ad.read_h5ad("lung_atlas.h5ad")

Next we filter to the single most common cell type and lung region. I've also created a `log_gene` column (only needed for the plots below).

In [ ]:
import scanpy as sc
import numpy as np

most_common_cell_type = atlas.obs['cell_type'].value_counts().idxmax()
most_common_location = atlas.obs['location'].value_counts().idxmax()
adata = atlas[(atlas.obs['cell_type'] == most_common_cell_type) & (atlas.obs['location'] == most_common_location)].copy()
adata.obs["log_gene"] = np.log(adata.obs["nGene"])

We'll work with just 1000 most highly variable genes.

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=1000, flavor='seurat_v3')
adata = adata[:, adata.var['highly_variable']].copy()

## Fit Model

These data were actually already normalized, so it's a little odd to fit a count model. We would alternatively use a `ZeroInflatedTruncatedGaussianCopula` but I found that to be too sensitive to outliers.

In [ ]:
from scdesigner.simulators import NegBinCopula
import scdiagnostics as scd

sim = NegBinCopula(mean_formula='~ donor_id * log_gene + np.log(nUMI)', dispersion_formula='~ donor_id * log_gene + np.log(nUMI)')
sim.fit(adata, lr=0.001, patience=10)

## Checks

We can use `scdiagnostics` to make sure the results are reasonable.

In [ ]:
samples = sim.sample()
#scd.compare_umap(adata, samples, color='log_gene')

In [ ]:
#scd.compare_umap(adata, samples, color='donor_id')

Here you can see that the original data were preprocessed because our NB data have to lie at counts (the $x$-axis has been log transformed).

In [ ]:
scd.compare_histogram(adata, samples)

In [ ]:
scd.compare_means(adata, samples)

This is not great... but since you'll be using another dataset, I'm not going to try to resolve this.

In [ ]:
scd.compare_variances(adata, samples)

## Pseudosamples

We don't have a method built directly into the simulator for generating pseudo-observations, but it's not too hard to do manually.

In [ ]:
import torch

z = []
for _, x in iter(sim.marginal.loader):
    x["group"] = torch.ones((len(x["mean"]), 1))
    z.append(sim.copula.pseudo_obs(x))

np.concat(z)

We do have a built-in method for getting the uniformized data though.

In [ ]:

u = []
for y, x in iter(sim.marginal.loader):
    u.append(sim.marginal.uniformize(y, x))

np.concat(u)